# 17 - CIDER T-cell Score vs Response, by CDR3 Read-Confidence Filter and Cohort

Methodological supplement: does restricting CIDER CDR3 calls to only
high-confidence (`PASS`) reads, or only low-confidence
(`NO_HIGH_QUAL_SUPPORT`) reads, change how well the CIDER T-cell Score
separates ICI Responders from Non-Responders — and is this consistent
across cancer types, not just pan-cancer? The primary (PASS +
low-confidence combined) pipeline is not re-plotted here since it's already
shown in the main Figure 3 panel; this supplement isolates the two
alternate-filter variants against each other.

**Structure:** two stacked panels — Panel A: PASS Only; Panel B:
Low-Confidence Only (`NO_HIGH_QUAL_SUPPORT`). The Panel B title spells out
the underlying CIDER `filter` value explicitly (rather than just
"Low-Confidence Only"), so the raw CIDER category being plotted is clear
without needing to check this notebook. Each panel is built exactly like
the main Figure 3 panel: a pooled "ICI-Treated" bar plus a per-cancer-type
breakdown (`cohort_broad_go`) on the x-axis, with x-axis labels dodged
across 2 lines to stay readable with this many cohorts.

**Output:** `figs/supplement_response_by_read_confidence.png`/`.rds`.

**Prerequisites:** `4_base.ipynb` (for `cpi_ready.rds` / `meta_base.csv`) and
`1b_cider_by_confidence.ipynb` (for the PASS-only / low-confidence-only
CIDER base variants) must have been run.


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

library(ggpubr)
library(patchwork)

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)


# 0 - Score each read-confidence variant

In [ ]:
# ── Primary pipeline (combined PASS + low-confidence) ────────────────────────
# Loaded exactly as 1_response.ipynb loads it for the main Figure 3 panel:
# one row per (sampleId, gp), *plus* a duplicated copy of every row with
# cohort_broad overwritten to "ICI-Treated" (added in 4_base.ipynb). We keep
# both copies here — no dedup — because that duplication is exactly what
# lets this panel show a pooled "ICI-Treated" bar alongside the
# per-cancer-type breakdown, each copy landing in its own cohort_broad_go
# bin below (same mechanism as Figure 3, not a double-count: the two copies
# never share an x-axis bin).
cpi_ready <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds"))

# ── Alternate-confidence CIDER base variants ─────────────────────────────────
# One row per (sampleId, gp); produced by 1b_cider_by_confidence.ipynb.
cider_pass    <- fread(paste0(CIDER_PROJ_DIR, "cider_base_pass_only.csv"))
cider_lowconf <- fread(paste0(CIDER_PROJ_DIR, "cider_base_low_confidence_only.csv"))

# Per-patient covariates needed to score + restrict to the ICI cohort with
# known response.
meta   <- fread(paste0(CIDER_PROJ_DIR, "meta_base.csv")) |>
  dplyr::select(sampleId, cpi, response, responder)
depths <- fread(paste0(COBALT_DIR, "depths.csv"))

# cohort_broad_go (pooled "ICI-Treated" + per-cancer-type, with per-group N)
# is computed once in 4_base.ipynb for cpi_ready. The PASS-only and
# low-confidence-only variants below cover exactly the same patients — the
# CDR3 filter only changes their *scores*, never who is ICI-treated or their
# cancer type — so the same cohort labels apply unchanged. Keeping both the
# real-cohort and pooled rows here (distinct(), not deduplicated to one row
# per sample) lets score_variant() below reproduce the same pooled-plus-
# per-cohort structure for those two variants.
cohort_lookup <- cpi_ready |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::select(sampleId, cohort_broad, cohort_broad_go, msi_high) |>
  dplyr::distinct()

# Split cohort_broad's residual "Other" bucket into Non-CRC MSI / Other
# (MSS) -- this panel reuses response_plot() with cohort_broad_go on the
# x-axis exactly like Figure 3's main panel (1_response.ipynb), so it gets
# the same split for the same reason: "Other" otherwise mixes MSI-high and
# microsatellite-stable patients from many different tumor types (see
# helpers/shortcuts.r::split_other_by_msi()). Level order:
# BROAD_LEVELS_OTHER_SPLIT (helpers/config.r). Display-only and local to
# this notebook's own cohort_lookup.
cohort_other_split_levels <- c(
  BROAD_LEVELS_OTHER_SPLIT,
  intersect("Other (Unknown MSI)",
            unique(split_other_by_msi(cohort_lookup$cohort_broad, cohort_lookup$msi_high)))
)
cohort_lookup <- cohort_lookup |>
  mutate(cohort_broad = factor(
    split_other_by_msi(cohort_broad, msi_high),
    levels = cohort_other_split_levels
  )) |>
  group_by(cohort_broad) |>
  mutate(n_broad = n_distinct(sampleId)) |>
  ungroup() |>
  mutate(
    cohort_broad_go = {
      lkp <- distinct(pick(cohort_broad, n_broad)) |> arrange(cohort_broad)
      factor(
        paste0(cohort_broad, " (N = ", n_broad, ")"),
        levels = paste0(lkp$cohort_broad, " (N = ", lkp$n_broad, ")")
      )
    }
  ) |>
  select(sampleId, cohort_broad_go)


In [ ]:
# Score one alternate-confidence variant: join to metadata + depth + cohort,
# restrict to ICI-treated patients with known RECIST response, compute the
# same depth-normalised score used everywhere else in the paper.
#
# The inner_join with cohort_lookup deliberately expands each patient to two
# rows (their real cancer type + the pooled "ICI-Treated" copy) — the same
# pooled-plus-per-cohort structure cpi_ready carries natively (see above),
# reproduced here explicitly since the CDR3-only base tables have no cohort
# information of their own.
score_variant <- function(cider_variant, label) {
  cider_variant |>
    dplyr::filter(gp == "T-cell") |>
    dplyr::inner_join(meta,          by = "sampleId") |>
    dplyr::inner_join(depths,        by = "sampleId") |>
    dplyr::inner_join(cohort_lookup, by = "sampleId") |>
    dplyr::filter(cpi, !is.na(response)) |>
    dplyr::mutate(
      cider_score_depth_adj = (cdr3_unique / sampleMedianDepth) * SCORE_SCALE,
      # "Non-Responder" before "Responder" -- matches the implicit
      # (unfactored, alphabetical) order response_plot()'s other callers get
      # for free (1_response.ipynb's Figure 3 panel, 19_dna_vs_rna_response.ipynb,
      # 21_tcell_geneset_panel.ipynb's Panel B), so this plot's box/legend
      # order is consistent with the rest of the paper's response boxplots.
      responder              = factor(responder, levels = c("Non-Responder", "Responder")),
      gp                     = "T-cell",
      read_filter            = label
    ) |>
    dplyr::select(sampleId, gp, responder, cider_score_depth_adj, read_filter, cohort_broad_go)
}

# The primary (PASS + low-confidence combined) pipeline is intentionally not
# scored/plotted here -- it's already shown in the main Figure 3 panel, and
# this supplement only needs the two alternate-filter variants side by side.
variant_data <- dplyr::bind_rows(
  score_variant(cider_pass,    "PASS Only"),
  score_variant(cider_lowconf, "Low-Confidence Only (NO_HIGH_QUAL_SUPPORT)")
) |>
  dplyr::mutate(read_filter = factor(read_filter, levels = c(
    "PASS Only", "Low-Confidence Only (NO_HIGH_QUAL_SUPPORT)"
  )))


# 1 - Figure

In [ ]:
# Two panels -- PASS Only and Low-Confidence Only (NO_HIGH_QUAL_SUPPORT) --
# each built exactly like Figure 3's main panel (response_plot() with
# cohort_broad_go on the x-axis), so both panels show the same
# pooled-"ICI-Treated"-plus-per-cancer-type comparison, just under a
# different CDR3 read-confidence filter. x_dodge = 2 spreads the cohort
# labels across two lines so they stay readable.
read_filter_levels <- levels(variant_data$read_filter)

read_confidence_panels <- lapply(read_filter_levels, function(lvl) {
  response_plot(
    dplyr::filter(variant_data, read_filter == lvl),
    "T-cell", "cohort_broad_go",
    title   = lvl,
    y_col   = "cider_score_depth_adj",
    x_dodge = 2
  )
})

p_response_by_confidence <- patchwork::wrap_plots(read_confidence_panels, ncol = 1, guides = "collect") +
  figure_annotation("CIDER T-cell Score vs ICI Response, by CDR3 Read-Confidence Filter and Cohort")

set_plot_dims(9, 8); p_response_by_confidence


In [ ]:
saveRDS(p_response_by_confidence, paste0(CIDER_PROJ_DIR, "figs/supplement_response_by_read_confidence.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_response_by_read_confidence.png"),
       p_response_by_confidence, width = 9, height = 8, dpi = 300)
message("Saved: supplement_response_by_read_confidence.rds and supplement_response_by_read_confidence.png")
